# Phase 5 MVP - Threshold Resist

End-to-end path: mask -> aerial image -> focus slice -> threshold resist -> CD/EPE.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().resolve()
if not (repo / "src").exists() and (repo.parent / "src").exists():
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src import constants as C
from src.dof import focus_drilling_average
from src.mask import MaskGrid, kirchhoff_mask, line_space_pattern
from src.metrics import critical_dimension, edge_positions, mean_absolute_epe, michelson_contrast
from src.pupil import PupilSpec
from src.resist_threshold import threshold_resist

In [ ]:
pitch_m = 80e-9
grid = MaskGrid(nx=512, ny=64, pixel_size=1e-9)
pattern = line_space_pattern(grid, pitch_m=pitch_m, duty_cycle=0.5)
mask = kirchhoff_mask(pattern)

aerial, wafer = focus_drilling_average(
    mask,
    grid,
    [0.0],
    pupil_spec=PupilSpec(grid_size=512, na=C.NA_HIGH, obscuration_ratio=0.0),
    anamorphic=False,
)
printed = threshold_resist(aerial, dose=1.0, threshold=0.2)

In [ ]:
cy = wafer.ny // 2
target_clear = pattern[cy, :] == 0.0
target_cd = critical_dimension(target_clear, grid.pixel_size)
printed_cd = critical_dimension(printed[cy, :], wafer.pixel_x_m)
target_edges = edge_positions(target_clear, grid.pixel_size)
printed_edges = edge_positions(printed[cy, :], wafer.pixel_x_m)

summary = {
    "target_cd_nm": target_cd * 1e9,
    "printed_cd_nm": printed_cd * 1e9,
    "cd_error_percent": abs(printed_cd - target_cd) / target_cd * 100.0,
    "mean_abs_epe_nm": mean_absolute_epe(target_edges, printed_edges) * 1e9,
    "contrast": michelson_contrast(aerial[cy, :]),
}
summary